# The International Brain Laboratory (2021) on DANDI:000045

Paper: The International Brain Laboratory, "Standardized and reproducible measurement of
decision-making in mice", eLife 10:e63711 (2021), doi:10.7554/eLife.63711.

Dataset: DANDI:000045, "IBL behavioral data", version **0.211209.1413**
(doi:10.48324/dandi.000045/0.211209.1413). The paper cites no version of the dandiset (its data
statement points to data.internationalbrainlab.org, and the dandiset names the bioRxiv preprint).
There are two published versions, 0.211209.1412 and 0.211209.1413, made a minute apart; they hold
the same files (checked below), so the most recent is pinned.

Claims: `claims/000045/10.7554-elife.63711.yaml`, written from the version of record (eLife,
PMC8137147) with its Methods, figure legends and Appendix 1, and the paper's code
(github.com/int-brain-lab/paper-behavior at commit edc4531). The claims file decides what is tested
and how; this notebook does not change it.

How claims are judged. Where the claims file lists several readings of a number, a claim matches
when it matches under any of them. The proficiency criterion P1 is run under four readings: the
claims file's two (1a criteria throughout; 1a, then 1b from September 2019) crossed with two
readings of "all contrasts introduced" that the claims file does not settle (a question is open).
When the outcome differs between the last two, the claim is blocked as `methods_underspecified`.
For a claim of absence, readings that give different outcomes also block it. Where the expected
values are not all met but some are, or the effect holds without the reported strength, the
outcome is `partially_reproduced`.

| Claim | Tested here |
|---|---|
| cohort-counts | tested |
| fig2g-days-to-proficiency | tested |
| fig2g-days-differ-across-labs | tested |
| fig2s1-trials-to-proficiency | tested |
| fig2h-proportion-learned-40-days | tested; blocked, the readings disagree |
| fig2bc-threshold-bias-iqr | tested |
| fig2-abs-bias-threshold-during-training | tested |
| fig2s2-variability-decreases | tested |
| early-sessions-predict-slow-learners | tested; blocked, the readings disagree |
| fig3c-easy-errors-basic | tested |
| fig3de-threshold-bias-basic | tested |
| fig3cde-no-lab-difference-medians | tested; blocked, the readings disagree |
| fig3cde-no-lab-difference-variances | tested |
| fig3s2a-trial-duration | tested |
| fig3s2b-trials-per-session | tested |
| fig3f-classifier-chance-basic | blocked: fewer than 8 mice in some labs |
| fig3f-positive-control | blocked: fewer than 8 mice in some labs |
| fig3s3-other-classifiers | blocked: fewer than 8 mice in some labs |
| lab-link (added: the labs and their time zones) | tested |
| fig3s1a-choice-consistency-basic | tested |
| fig3s1bc-choice-consistency-full | tested |
| fig3s4-looser-criterion | tested (classifier part not run) |
| fig3s5-first-full-sessions | tested (classifier part not run) |
| fig3s5a-prior-used-from-first-sessions | tested |
| full-task-training-time | tested |
| fig4cd-biasshift-largest-at-zero | tested |
| fig4de-biasshift-no-lab-difference | tested |
| fig4f-threshold-unchanged-by-block | tested |
| fig4gh-lapses-change-with-block | tested |
| fig4i-bias-shifts-with-block | tested |
| fig4fi-no-lab-difference | tested |
| fig4j-classifier-chance-full | blocked: fewer than 8 mice in some labs |
| fig5cf-glm-weights | tested |
| fig5dg-history-weights-higher-full | tested |
| fig5s2-glm-accuracy | tested |
| fig5s2-weights-similar-across-labs | tested |
| fig5s1b-win-stay-lose-switch | tested |
| fig5s1c-updating-by-contrast | tested |
| session-end-criteria | tested |

The notebook streams every file (about five minutes) and runs in about twenty minutes in all.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import ibl_behavior as ib

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
T0 = time.time()
print("dandiset", ib.DANDISET_ID, "version", ib.DANDISET_VERSION)

## Loading

Every file of the pinned version is streamed; only the trials table, the session metadata and
the rig settings are read. This takes about six minutes.

In [ ]:
#LOAD
meta = ib.version_metadata()
assets = ib.list_assets()
other = ib.list_assets("0.211209.1412")
print(meta["doi"], "| assets", len(assets), "| subjects (archive summary)",
      meta["assetsSummary"]["numberOfSubjects"])
print("the two published versions hold the same files:",
      sorted(assets.blob) == sorted(other.blob) and sorted(assets.path) == sorted(other.path))
sessions = ib.read_all(assets.asset_id.tolist())
errors = [s for s in sessions if "error" in s]
print("files read", len(sessions), "| failed", len(errors))
sessions = [s for s in sessions if "error" not in s]
S = ib.session_table(sessions)
T = ib.trials_frame(sessions)
print(f"loaded in {(time.time() - T0) / 60:.1f} min")

## Data check

What was loaded, and each check of the description against the data.

In [ ]:
import hashlib
print("sessions", len(S), "| with trials", (S.n_trials > 0).sum(), "| trials", len(T))
print("subjects", S.subject_id.nunique(), "| nicknames", S.nickname.nunique(),
      "| subject folder = subject_id in file:", (assets.set_index("asset_id").subject_dir
                                                  .reindex(S.asset_id).to_numpy() == S.subject_id).all())
print("dates", S.date.min(), "to", S.date.max())
print("\nsessions by institution and lab number (code numbering by institution)")
print(S.groupby(["institution", "lab_number"]).agg(sessions=("session_id", "size"),
                                                    mice=("subject_id", "nunique")))
print("\nsessions by task protocol")
print(S.protocol.str.replace(r"[\d.]+$", "", regex=True).value_counts())
print("\nsessions by task as used here:", S.task.value_counts().to_dict())
print("sex of mice:", S.groupby("subject_id").sex.first().value_counts().to_dict())

# duplicated recordings: identical choices, contrasts and stimulus times
key = {}
for s in sessions:
    if s.get("choice") is None or len(s["choice"]) < 20:
        continue
    h = hashlib.md5(np.ascontiguousarray(s["choice"]).tobytes()
                    + np.ascontiguousarray(s["contrastLeft"]).tobytes()
                    + np.ascontiguousarray(s["stimOn_times"]).tobytes()).hexdigest()
    key.setdefault(h, []).append(s["session_id"])
dups = [v for v in key.values() if len(v) > 1]
print("\nrecordings deposited more than once:", len(dups))
for v in dups:
    print(S.set_index("session_id").loc[v, ["nickname", "start", "n_trials", "protocol"]])
dup_ids = {x for v in dups for x in v}
print("excluded session of the code present:", (S.session_id == ib.EXCLUDED_SESSION).any())

In [ ]:
# Trial variables: values, ranges and units, against the column descriptions
print("contrastLeft/Right values (fraction):", sorted(set(np.unique(T.contrastLeft.dropna()))
                                                     | set(np.unique(T.contrastRight.dropna()))))
print("trials with both or neither contrast set:",
      int((T.contrastLeft.notna() == T.contrastRight.notna()).sum()))
print("signed contrast (%) values:", sorted(T.signed_contrast.unique()))
odd = T[T.abs_contrast.isin([4.0, 8.0])].session_id.unique()
print("sessions with 4% or 8% contrasts:", S[S.session_id.isin(odd)][["nickname", "date", "protocol"]].to_dict("records"))
print("probabilityLeft values:", T.probabilityLeft.round(3).value_counts().to_dict())
tt = T.merge(S[["session_id", "task"]], on="session_id")
tb = tt[tt.task == "training"]
print("basic-task sessions with probabilityLeft other than 0.5 on some trial: %d of %d; trials %.1f%%"
      % ((tb.probabilityLeft != 0.5).groupby(tb.session_id).any().sum(), tb.session_id.nunique(),
         100 * (tb.probabilityLeft != 0.5).mean()))
print("basic task, fraction of left stimuli by probabilityLeft:",
      tb.groupby(tb.probabilityLeft.round(1)).contrastLeft.apply(lambda x: round(x.notna().mean(), 3)).to_dict())
del tt, tb
print("choice values:", T.choice.value_counts().to_dict(), "| feedbackType:", T.feedbackType.value_counts().to_dict())
print("no-response trials (choice 0): %.2f%%" % (100 * (T.choice == 0).mean()))
# choice convention: on trials with a stimulus on one side, which choice is rewarded
t = T[(T.choice != 0) & (T.abs_contrast > 0)]
print("\nrewarded fraction by stimulus side and choice:")
print(t.groupby([np.where(t.signed_contrast > 0, "right stim", "left stim"), "choice"]).correct.mean().round(4))
print("-> choice -1 (CCW) is the rightward choice, as coded in trials_frame")
q = T.rt.quantile([0, 0.01, 0.5, 0.99, 1]).round(3).to_dict()
print("\ntrial duration, response - stimOn (s), quantiles:", q)
print("negative trial durations:", int((T.rt < 0).sum()), "| missing stimOn:", int(T.stimOn_times.isna().sum()),
      "| missing response time:", int(T.response_times.isna().sum()))
miss = T.stimOn_times.isna().groupby(T.session_id).mean()
print("sessions with some trials lacking stimOn: %d; with none: %d" % ((miss > 0).sum(), (miss == 1).sum()))
print("sessions with a negative trial duration: %d" % (T.rt < 0).groupby(T.session_id).any().sum())
print("trials per session: median", S.n_trials.median(), "range", S.n_trials.min(), "-", S.n_trials.max())
print("session duration (min, end - start): median %.1f, range %.1f - %.1f, missing end %d"
      % (S.duration_min.median(), S.duration_min.min(), S.duration_min.max(), S.duration_min.isna().sum()))
# session_start_time is stated in UTC (+00:00); start hours by lab
hour = pd.to_datetime(S.start.str[:19]).dt.hour
print("\nsession_start_time offsets:", S.start.str[19:].value_counts().to_dict())
print("start hour by lab (5th, 50th, 95th percentile):")
print(S.assign(hour=hour).groupby("lab_number").hour.quantile([0.05, 0.5, 0.95]).unstack())

In [ ]:
# Session-end status in the rig settings: SUBJECT_DISENGAGED_TRIGGERED and _TRIALNUM
print("end status (SUBJECT_DISENGAGED_TRIGGERED):", S.disengaged.astype(str).value_counts().to_dict())
x = S[S.disengaged_trial.notna()]
print("sessions with a trigger trial:", len(x), "| trigger trial after the last trial:",
      int((x.disengaged_trial > x.n_trials).sum()))
# Wheel series: data and timestamps should have the same length
wm = S[S.wheel_n.notna() & (S.wheel_n != S.wheel_ts_n)]
print("files without a wheel series:", int(S.wheel_n.isna().sum()), "| wheel data and timestamps of different length:",
      len(wm), "files;", wm.lab_number.value_counts().sort_index().to_dict())
# Clock check: wheel position against the trial table, in 16 sessions drawn at random.
# The wheel displacement from stimulus onset to response should follow the choice when
# the trial times are taken as they are, and not when they are shifted by 30 s.
rng_check = np.random.default_rng(1)
pick = S[(S.n_trials > 300)].sample(16, random_state=1)
res = ib.wheel_many(pick.asset_id.tolist(), [0.0, 30.0, -30.0])
rows = []
for (sid, nick, lab), (disp, ch) in zip(pick[["session_id", "nickname", "lab_number"]].to_numpy(), res):
    if disp is None:
        rows.append(dict(session=sid[:8], mouse=nick, lab=lab, ok=False, reason=ch))
        continue
    m = ch != 0
    r = {f"agree_{int(s):+d}s": np.nanmean(np.sign(disp[s][m]) == -ch[m]) for s in disp}
    rows.append(dict(session=sid[:8], mouse=nick, lab=lab, ok=True, **r))
wheel_check = pd.DataFrame(rows)
print(wheel_check.round(3).to_string())
print("agreement is the fraction of trials whose wheel displacement has the sign of -choice; 0 means the opposite sign")
print("median agreement, true times %.3f; shifted +30 s %.3f; -30 s %.3f"
      % tuple(wheel_check[c].median() for c in ["agree_+0s", "agree_+30s", "agree_-30s"]))

### Selection and proficiency

The paper's mice are those of project `ibl_neuropixel_brainwide_01` (claims file header). The
dataset begins on 2019-06-11, while the paper's mice began training in January 2019 (Methods), so
only mice whose first session in the dataset is a first-stage session (50% and 100% contrasts
only) have their whole training in the dataset; the training-time claims need that. The files
carry no training status, so basic (P1) and full (P2) task proficiency are recomputed from the
trials with the criteria of Appendix 1, table 1d (claims file header). P1 has four readings: the
claims file's two (1a criteria throughout; 1a, then 1b from September 2019) crossed with two
readings of "all contrasts introduced", which the claims file does not settle (each of the three
sessions, or the three together; a question is open).

In [ ]:
S = S[~S.session_id.isin(dup_ids) & (S.n_trials > 0)].copy()
T = T[T.session_id.isin(S.session_id)].copy()
mice = ib.project_mice(S)
lab = S.groupby("subject_id").lab_number.first()
nick = S.groupby("subject_id").nickname.first()
print("mice in the dataset", S.subject_id.nunique(), "| of the project", len(mice))
print("not in the project:", list(nick[~nick.index.isin(mice)]))
Dall = ib.daily(T, S)
first = Dall.groupby("subject_id").head(1).set_index("subject_id")
first_stage = first.contrasts.apply(lambda c: c == frozenset([50.0, 100.0]))
complete = first_stage[first_stage].index.intersection(mice)
print("project mice whose first session is a first-stage session:", len(complete),
      "| others:", len(mice) - len(complete))
print("first session of the others: task", S[S.subject_id.isin(mice.difference(complete))]
      .sort_values("start").groupby("subject_id").task.first().value_counts().to_dict(),
      "| date", S[S.subject_id.isin(mice.difference(complete))].groupby("subject_id").date.min()
      .value_counts().head(3).to_dict())
print("complete-history mice by lab:", lab[complete].value_counts().sort_index().to_dict())
Sc = S[S.subject_id.isin(complete)]
Tc = T[T.session_id.isin(Sc.session_id)]
Dc = Dall[Dall.subject_id.isin(complete)]
Dtr = ib.daily(Tc, Sc, ["training"])
Dfull = ib.daily(Tc, Sc, ["biased"])
Tm = ib.trials_with_days(Tc, Sc, Dc)
print("days with more than one session:", int((Sc.groupby(["subject_id", "date"]).size() > 1).sum()))

In [ ]:
READINGS = [("1a", "each"), ("1a", "pooled"), ("1a1b", "each"), ("1a1b", "pooled")]
RN = {r: f"{r[0]}/{r[1]}" for r in READINGS}
groups_tr = ib.trial_groups(Tc, Sc, "training")
P1 = {r: ib.basic_proficiency(Dtr, Tc, Sc, *r, groups=groups_tr) for r in READINGS}
P2 = ib.full_proficiency(Dfull, Tc, Sc)
first_biased = Dc[Dc.tasks.apply(lambda t: "biased" in t)].groupby("subject_id").tday0.min()
rows = []
for r in READINGS:
    p = P1[r].p1_tday0
    gap = (first_biased - p).dropna()
    rows.append(dict(reading=RN[r], P1=len(p), **{f"P1 {l}": int((lab[p.index] == l).sum()) for l in ib.LABS},
                     first_full_day_minus_P1_median=gap.median(),
                     within_1_to_3_days=f"{int(gap.between(1, 3).sum())}/{len(gap)}",
                     full_task_without_P1=int(len(first_biased.index.difference(p.index)))))
prof = pd.DataFrame(rows).set_index("reading")
print(prof.to_string())
print("\nP2 (full task proficiency):", len(P2), "mice;", lab[P2.index].value_counts().sort_index().to_dict())
print("P2 mice without P1 under each reading:", {RN[r]: len(P2.index.difference(P1[r].index)) for r in READINGS})
# Sensitivity: trials counted as completed only when the mouse responded
Dtr_resp = ib.daily(Tc[Tc.choice != 0], Sc, ["training"])
for r in READINGS:
    alt = ib.basic_proficiency(Dtr_resp, Tc, Sc, *r, groups=groups_tr)
    j = P1[r][["p1_tday0"]].join(alt[["p1_tday0"]], rsuffix="_resp", how="outer")
    print(f"{RN[r]}: P1 differs for {int((j.p1_tday0 != j.p1_tday0_resp).sum())} mice when only responded trials count")
# 1b from the end of September 2019 in place of the first
ib_date = ib.DATE_1B
ib.DATE_1B = "2019-09-30"
for r in READINGS[2:]:
    alt = ib.basic_proficiency(Dtr, Tc, Sc, *r, groups=groups_tr)
    print(f"{RN[r]} with 1b from 2019-09-30: {len(alt)} mice reach P1 (against {len(P1[r])})")
ib.DATE_1B = ib_date

## cohort-counts

> Of 206 mice that started training, 140 reached proficiency in the basic task and 98 in the
> full task.

Counts of distinct mice under the paper's selection; proficiency as P1 (four readings) and P2.

In [ ]:
print("project mice in the dataset:", len(mice), "| with their whole training in the dataset:", len(complete))
print("P1 by reading:", {RN[r]: len(P1[r]) for r in READINGS})
print("P2:", len(P2))
exp = [(206, 20.6), (140, 14), (98, 9.8)]
for name, n in [("started training, whole training in the dataset", len(complete)),
                ("project mice with any session", len(mice))]:
    print(f"{name}: {n}, within 206 +- 20.6: {abs(n - 206) <= 20.6}")
for r in READINGS:
    print(f"P1 {RN[r]}: {len(P1[r])}, within 140 +- 14: {abs(len(P1[r]) - 140) <= 14}")
print(f"P2: {len(P2)}, within 98 +- 9.8: {abs(len(P2) - 98) <= 9.8}")

**Result: not reproduced.** The dataset holds 177 mice of the paper's project, and 136 of them
have their whole training in it, against 206 that started training. 75 to 84 of these reach basic
task proficiency under the 1a readings (52 to 58 under 1a/1b), against 140, and 60 reach full
task proficiency, against 98. No count is within tolerance under any reading. The dataset holds
only sessions from 2019-06-11 on, while training began in January 2019.

## fig2g-days-to-proficiency

> Basic task training took 18.4 +- 13.0 days (s.d., n = 140 mice), from 3 days for the fastest
> mouse to 59 days for the slowest.

Training days up to P1, counting the P1 day (code query branch) or not (CSV branch).

In [ ]:
def check(vals, exp):
    return all(abs(v - e) <= tol for v, (e, tol) in zip(vals, exp))

EXP_DAYS = [(18.4, 1.84), (13.0, 1.3), (3, 0.3), (59, 5.9)]
rows = []
days = {}
for r in READINGS:
    for cnt in ("incl", "excl"):
        d = P1[r].p1_tday0 + (1 if cnt == "incl" else 0)
        days[(r, cnt)] = d
        v = [d.mean(), d.std(), d.min(), d.max()]
        rows.append(dict(reading=RN[r], count=cnt, n=len(d), mean=v[0], sd=v[1], min=v[2], max=v[3],
                         n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_DAYS)),
                         all_in_tol=check(v, EXP_DAYS)))
fig2g = pd.DataFrame(rows)
print(fig2g.round(2).to_string())
d = days[(READINGS[0], "incl")]
print("\nby lab (1a/each, incl):")
print(d.groupby(lab[d.index]).describe()[["count", "mean", "50%", "min", "max"]].round(1))

**Result: partially reproduced.** Over the 52 to 84 mice that reach P1, the mean is 17.3 to 21.7
days, within 18.4 +- 1.84 under the 1a readings (18.99 days for 1a/each, counting the P1 day). The
s.d. is 10.5 days under the 1a readings and 14.6 under 1a/1b, against 13.0 +- 1.3; no reading has
all four values within tolerance (at most two). The fastest mouse takes 3 to 5 days and the slowest
53 to 58, against 3 and 59.

## fig2g-days-differ-across-labs

> The number of days to basic task proficiency differed across laboratories; within-lab
> interquartile ranges ranged from 8 days (Lab 2) to 22 days (Lab 6).

In [ ]:
rows = []
for (r, cnt), d in days.items():
    l = lab[d.index]
    iqr = d.groupby(l).quantile(0.75) - d.groupby(l).quantile(0.25)
    p_kw = ib.lab_test(d, l, "kruskal")
    p_code = ib.lab_test(d, l, "code")
    rows.append(dict(reading=RN[r], count=cnt, n=len(d), p_kruskal=p_kw, p_code_switch=p_code,
                     iqr_lab2=iqr.get("Lab 2"), iqr_lab6=iqr.get("Lab 6"),
                     iqr_range=f"{iqr.min():.1f}-{iqr.max():.1f}",
                     p_below_0001=p_kw < 0.001,
                     iqr_in_tol=abs(iqr.get("Lab 2", np.nan) - 8) <= 0.8 and abs(iqr.get("Lab 6", np.nan) - 22) <= 2.2))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))
print("mice per lab (1a/each):", lab[days[(READINGS[0], 'incl')].index].value_counts().sort_index().to_dict())

**Result: not reproduced.** The Kruskal-Wallis p across labs is 0.024 to 0.25 (0.079 for 1a/each),
above the paper's p < 0.001 under every reading. The Lab 2 interquartile range is 5 to 9 days (8
under 1a/pooled) and the Lab 6 range 5 to 25 days, against 8 and 22; Lab 6 is within tolerance
only under the 1a/1b readings (24.75), where Lab 2 is not.

## fig2s1-trials-to-proficiency

> Measured in trials, basic task training took 10.8 +- 8.6 thousand trials (s.d., n = 140), from
> one thousand for the fastest mouse to 43 thousand for the slowest; reaching 80% correct on easy
> trials took on average about 7.4 thousand trials.

In [ ]:
EXP_TR = [(10800, 1080), (8600, 860), (1000, 100), (43000, 4300)]
rows = []
for (r, cnt), d in days.items():
    dd = Dc[Dc.subject_id.isin(d.index)]
    lim = dd.subject_id.map(d)
    # days counted: tday0 < days (excl) or <= p1 day (incl); d is the count, so tday0 < d
    tr = dd[dd.tday0 < lim].groupby("subject_id").n_trials.sum()
    v = [tr.mean(), tr.std(), tr.min(), tr.max()]
    rows.append(dict(reading=RN[r], count=cnt, n=len(tr), mean=v[0], sd=v[1], min=v[2], max=v[3],
                     n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_TR)), all_in_tol=check(v, EXP_TR)))
print(pd.DataFrame(rows).round(0).to_string())

def trials_to_80(mice_idx, variant=False):
    dd = Dc[Dc.subject_id.isin(mice_idx) & (Dc.n_trials > 100)].sort_values(["subject_id", "tday0"]).copy()
    dd["perf_f"] = dd.groupby("subject_id").perf_easy.transform(
        lambda x: x.rolling(3, center=True, min_periods=1).median())
    dd["day"] = dd.groupby("subject_id").cumcount()
    dd["cum"] = dd.groupby("subject_id").n_trials.cumsum()
    if variant:
        f = dd[dd.perf_f > 80].groupby("subject_id").cum.first()
        return f.mean(), len(f)
    g = dd.groupby("day").agg(perf=("perf_f", "mean"), cum=("cum", "mean"), n=("subject_id", "size"))
    hit = g[g.perf > 80]
    return (hit.cum.min() if len(hit) else np.nan), int(hit.index.min()) if len(hit) else None

for r in READINGS:
    v, day = trials_to_80(P1[r].index)
    vv, nn = trials_to_80(P1[r].index, variant=True)
    print(f"{RN[r]}: trials to a mean of 80% on easy trials {v:.0f} (first at training day {day}); "
          f"within 7400 +- 740: {abs(v - 7400) <= 740} | variant, mean of each mouse's first: {vv:.0f} (n={nn})")

**Result: partially reproduced.** The mean number of trials to P1 is 10,040 to 13,502 and within
10,800 +- 1,080 under the 1a readings (11,235 for 1a/each, counting the P1 day). The s.d. (6,929 to
8,900), the fastest mouse (1,195 to 2,438 trials) and the slowest (32,235 to 34,998) are outside
tolerance under every reading. The trial count at which the mean easy-trial performance first
exceeds 80% is 7,302 under 1a/pooled, within 7,400 +- 740, and 5,535 to 6,603 under the others.

## fig2h-proportion-learned-40-days

> About 80% of mice that started training would learn the basic task within the 40 days usually
> allotted (Kaplan-Meier estimate over all 206 mice).

In [ ]:
from lifelines import KaplanMeierFitter
total_days = Dc.groupby("subject_id").tday0.max() + 1
last_date = Dc.groupby("subject_id").date.max()
rows = []
for r in READINGS:
    for cnt in ("incl", "excl"):
        for variant in ("main", "drop_in_training"):
            idx = complete
            p = days[(r, cnt)]
            dur = pd.Series(np.where(idx.isin(p.index), p.reindex(idx), total_days.reindex(idx)), index=idx)
            ev = pd.Series(idx.isin(p.index), index=idx)
            if variant == "drop_in_training":
                # still in training at the cutoff: no P1 and a session in the last 7 days before it
                still = (~ev) & (last_date.reindex(idx) >= "2020-03-16")
                dur, ev = dur[~still], ev[~still]
            km = KaplanMeierFitter().fit(dur, ev)
            at40 = 1 - km.predict(40)
            atlast = 1 - km.predict(dur.max())
            rows.append(dict(reading=RN[r], count=cnt, variant=variant, n=len(dur), events=int(ev.sum()),
                             p40=at40, p_last=atlast, last_day=dur.max(),
                             ok=abs(at40 - 0.8) <= 0.08 and atlast > at40))
print(pd.DataFrame(rows).round(3).to_string())

**Result: blocked (methods_underspecified).** With every mouse that started training (136, mice
without P1 censored), the Kaplan-Meier proportion at day 40 is 0.751 to 0.759 under 1a/pooled,
within 0.8 +- 0.08 and lower than at the last day (0.91), but 0.670 to 0.678 under 1a/each and
0.42 to 0.47 under the 1a/1b readings. The outcome turns on whether "all contrasts introduced" is
required of each session or of the three together, which a question asks. In the code's variant,
leaving out mice still in training at the cutoff, both 1a readings are within tolerance (0.742 to
0.822).

## fig2bc-threshold-bias-iqr

> Contrast threshold and bias varied little during training; the threshold interquartile range
> slightly decreased from 12-29% in the first 10 training days to 11-23% thereafter, and the bias
> interquartile range went from -5.2 to 7.1% to -5.6 to 6.5%.

Per training day (all sessions of the day pooled), 50:50 days with more than 100 trials and six
or more signed contrasts are fitted. Training day counted from 0 at the mouse's first session.
50:50 days are days with basic-task sessions only: in basic-task sessions `probabilityLeft` varies
from trial to trial (see the data check), so it cannot mark them.

In [ ]:
keep = Dc[(Dc.n_trials > 100) & Dc.basic_only & (Dc.n_signed >= 6)]
tk = Tm.merge(keep[["subject_id", "date"]], on=["subject_id", "date"])
day_fits = ib.mouse_fits(tk, by=("subject_id", "tday0")).reset_index()
day_fits = day_fits.sort_values(["subject_id", "tday0"])
for v in ("threshold", "bias"):
    day_fits[v + "_f"] = day_fits.groupby("subject_id")[v].transform(
        lambda x: x.rolling(3, center=True, min_periods=1).median())
print("day fits:", len(day_fits), "of", len(keep), "days;", day_fits.threshold.isna().sum(), "without a fit")
print("threshold range %.1f-%.1f, bias range %.1f-%.1f" % (day_fits.threshold.min(), day_fits.threshold.max(),
                                                            day_fits.bias.min(), day_fits.bias.max()))
EXP_BC = [(12, 1.2), (29, 2.9), (11, 1.1), (23, 2.3), (-5.2, 0.52), (7.1, 0.71), (-5.6, 0.56), (6.5, 0.65)]
rows = []
for r in READINGS:
    f = day_fits[day_fits.subject_id.isin(P1[r].index)]
    for filt in ("filtered", "unfiltered"):
        sfx = "_f" if filt == "filtered" else ""
        early = f[f.tday0 <= 9]
        for after in ("10-39", "10+"):
            late = f[(f.tday0 >= 10) & ((f.tday0 <= 39) if after == "10-39" else True)]
            v = [early["threshold" + sfx].quantile(.25), early["threshold" + sfx].quantile(.75),
                 late["threshold" + sfx].quantile(.25), late["threshold" + sfx].quantile(.75),
                 early["bias" + sfx].quantile(.25), early["bias" + sfx].quantile(.75),
                 late["bias" + sfx].quantile(.25), late["bias" + sfx].quantile(.75)]
            narrower = (v[3] - v[2]) < (v[1] - v[0])
            rows.append(dict(reading=RN[r], filt=filt, after=after, n_early=len(early), n_late=len(late),
                             thr_early=f"{v[0]:.1f}-{v[1]:.1f}", thr_late=f"{v[2]:.1f}-{v[3]:.1f}",
                             bias_early=f"{v[4]:.1f}-{v[5]:.1f}", bias_late=f"{v[6]:.1f}-{v[7]:.1f}",
                             n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_BC)),
                             narrower=narrower, ok=check(v, EXP_BC) and narrower))
fig2bc = pd.DataFrame(rows)
print(fig2bc.to_string())

**Result: partially reproduced.** Under every reading the threshold interquartile range after day
10 is narrower than in days 1 to 10 (for 1a/each, filtered, days 10 to 39: 11.4-22.4% against
10.7-24.5%). No reading puts all eight bounds within tolerance; at most four (1a/pooled, filtered:
threshold 11.1-24.6% then 11.7-23.8%, bias -8.3 to 6.5% then -8.8 to 7.6%). The early threshold
range is narrower than the paper's 12-29%, and the bias ranges are shifted to the left.

## fig2-abs-bias-threshold-during-training

> During training mice were not systematically biased to one side; absolute bias averaged
> 9.8 +- 12.1 (s.d.), and contrast threshold, which stabilized after about 10 sessions, averaged
> 17.8 +- 11.7%.

In [ ]:
EXP_AB = [(9.8, 0.98), (12.1, 1.21), (17.8, 1.78), (11.7, 1.17)]
rows = []
for r in READINGS:
    f = day_fits[day_fits.subject_id.isin(P1[r].index)].copy()
    f["lab"] = lab[f.subject_id].to_numpy()
    for filt in ("filtered", "unfiltered"):
        sfx = "_f" if filt == "filtered" else ""
        ab = f["bias" + sfx].abs()
        for after in ("10-39", "10+"):
            late = f[(f.tday0 >= 10) & ((f.tday0 <= 39) if after == "10-39" else True)]
            th = late["threshold" + sfx]
            for avg in ("pooled", "lab means"):
                if avg == "pooled":
                    m_ab, m_th = ab.mean(), th.mean()
                else:
                    m_ab = ab.groupby(f.lab).mean().mean()
                    m_th = th.groupby(late.lab).mean().mean()
                v = [m_ab, ab.std(), m_th, th.std()]
                rows.append(dict(reading=RN[r], filt=filt, after=after, avg=avg, abs_bias=v[0], abs_bias_sd=v[1],
                                 thr=v[2], thr_sd=v[3],
                                 means_ok=abs(v[0] - 9.8) <= 0.98 and abs(v[2] - 17.8) <= 1.78,
                                 sds_ok=abs(v[1] - 12.1) <= 1.21 and abs(v[3] - 11.7) <= 1.17))
print(pd.DataFrame(rows).round(2).to_string())
f = day_fits[day_fits.subject_id.isin(P1[READINGS[0]].index)]
print("\nsigned bias over all days (1a/each, filtered): mean %.2f, median %.2f; per lab mean:"
      % (f.bias_f.mean(), f.bias_f.median()),
      f.bias_f.groupby(lab[f.subject_id].to_numpy()).mean().round(2).to_dict())

**Result: reproduced.** Without the median filter, the mean absolute bias over all days is 9.66
(1a/each) and 9.81 (1a/pooled), and the mean threshold after day 10 is 18.03 and 18.61%, both
within tolerance of 9.8 and 17.8, for both readings of "all contrasts introduced". The s.d.s are
smaller than the paper's under every reading (absolute bias 6.1 to 8.5 against 12.1; threshold 8.0
to 9.8 against 11.7). The mean signed bias is -1.50 (1a/each, filtered), with lab means from -5.97
to 2.35; the paper's "similar across laboratories" is not tested by the criterion.

## fig2s2-variability-decreases

> Variability in performance across mice decreased with training but did not disappear; between
> training days 15 and 40, mean easy-trial performance rose from 80.7% to 91.1% while its s.d.
> across mice fell from 19.1% to 10.1%.

In [ ]:
EXP_V = [(80.7, 8.07), (91.1, 9.11), (19.1, 1.91), (10.1, 1.01)]

def perf_by_day(mice_idx, only_5050):
    dd = Dc[Dc.subject_id.isin(mice_idx) & (Dc.n_trials > 100)]
    if only_5050:
        dd = dd[dd.basic_only]
    dd = dd.sort_values(["subject_id", "tday0"]).copy()
    dd["perf_f"] = dd.groupby("subject_id").perf_easy.transform(
        lambda x: x.rolling(3, center=True, min_periods=1).median())
    dd["day"] = dd.groupby("subject_id").cumcount()
    return dd.groupby("day").perf_f.agg(["mean", "std", "size"])

rows = []
for r in READINGS:
    idx = P1[r].index
    for name, g in (("(1) all sessions", perf_by_day(idx, False)), ("(2) basic-task days only", perf_by_day(idx, True))):
        for a, b in ((15, 40), (14, 39)):
            if b not in g.index:
                continue
            v = [g.loc[a, "mean"], g.loc[b, "mean"], g.loc[a, "std"], g.loc[b, "std"]]
            rows.append(dict(reading=RN[r], data=name, days=f"{a},{b}", n=f"{g.loc[a, 'size']},{g.loc[b, 'size']}",
                             mean_a=v[0], mean_b=v[1], sd_a=v[2], sd_b=v[3],
                             n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_V)),
                             ok=check(v, EXP_V) and v[1] > v[0] and v[3] < v[2]))
    # (3) unfiltered per-day performance, days strictly within 2 of the bin centre; mean per mouse in the bin
    dd = Dc[Dc.subject_id.isin(idx)]
    vals = []
    for c in (15, 39):
        b = dd[(dd.tday0 - c).abs() < 2].groupby("subject_id").perf_easy.mean()
        vals.append(b)
    v = [vals[0].mean(), vals[1].mean(), vals[0].std(), vals[1].std()]
    rows.append(dict(reading=RN[r], data="(3) binned", days="15,39", n=f"{len(vals[0])},{len(vals[1])}",
                     mean_a=v[0], mean_b=v[1], sd_a=v[2], sd_b=v[3],
                     n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_V)),
                     ok=check(v, EXP_V) and v[1] > v[0] and v[3] < v[2]))
print(pd.DataFrame(rows).round(2).to_string())

**Result: partially reproduced.** Under every reading the mean easy-trial performance rises and its
s.d. across mice falls between days 15 and 40 (1a/each, all sessions: 87.22 to 91.23%, s.d. 13.54
to 7.77%). The two means are within tolerance under most readings, but no reading has the s.d.s
within tolerance (13.1 to 16.5% at day 15 against 19.1%; 3.7 to 7.8% at day 40 against 10.1%).

## early-sessions-predict-slow-learners

> A Random Forests decoder applied to the change in easy-trial performance over the first five
> sessions predicted whether a mouse would be in the bottom quartile of learning speed with 53%
> accuracy (chance 25%), and misclassified a top-quartile mouse as bottom-quartile only 7% of the
> time.

The readings of the criterion: features (a) four differences, (b) five performances; labels
(A) four quartiles, (B) bottom quartile against the rest; quartiles from days to P1 or trials to
P1; accuracy (i) recall of the bottom quartile, (ii) four-class accuracy under (A), (iii)
precision of the bottom quartile. scikit-learn RandomForestClassifier with default settings,
leave-one-out; its seed is a standard choice (0).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
first5 = Dc[Dc.tday0 <= 4].pivot(index="subject_id", columns="tday0", values="perf_easy")
rows = []
for r in READINGS:
    for qsrc in ("days", "trials"):
        d = days[(r, "incl")]
        if qsrc == "trials":
            dd = Dc[Dc.subject_id.isin(d.index)]
            d = dd[dd.tday0 < dd.subject_id.map(d)].groupby("subject_id").n_trials.sum()
        idx = d.index.intersection(first5.dropna().index)
        q = pd.qcut(d[idx].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)  # 4 = slowest
        F = {"a": np.diff(first5.loc[idx].to_numpy(), axis=1), "b": first5.loc[idx].to_numpy()}
        for feat, X in F.items():
            for lab_kind in ("A", "B"):
                y = q.to_numpy() if lab_kind == "A" else (q.to_numpy() == 4).astype(int)
                pred = cross_val_predict(RandomForestClassifier(random_state=0, n_jobs=4), X, y, cv=LeaveOneOut())
                bottom = 4 if lab_kind == "A" else 1
                recall = np.mean(pred[y == bottom] == bottom)
                precision = np.mean(y[pred == bottom] == bottom) if (pred == bottom).any() else np.nan
                acc4 = np.mean(pred == y) if lab_kind == "A" else np.nan
                top_as_bottom = np.mean(pred[q.to_numpy() == 1] == bottom)
                rows.append(dict(reading=RN[r], quartiles=qsrc, features=feat, labels=lab_kind, n=len(idx),
                                 recall=recall, four_class=acc4, precision=precision, top_as_bottom=top_as_bottom,
                                 base_rate=np.mean(y == bottom)))
es = pd.DataFrame(rows)
print(es.round(3).to_string())
acc = pd.concat([es.recall, es.four_class.dropna(), es.precision])
print("\naccuracy readings within 0.53 +- 0.053: %d of %d; range %.3f-%.3f"
      % (int(((acc - 0.53).abs() <= 0.053).sum()), len(acc), acc.min(), acc.max()))
print("top quartile classed as bottom, range %.3f-%.3f; within 0.07 +- 0.007: %d of %d"
      % (es.top_as_bottom.min(), es.top_as_bottom.max(), int(((es.top_as_bottom - 0.07).abs() <= 0.007).sum()), len(es)))

**Result: blocked (methods_underspecified).** The readings disagree on whether the accuracy is
near 53%: 3 of 80 accuracy values are within 0.53 +- 0.053, and the others run from 0.000 to
0.476. The share of top-quartile mice classed as bottom quartile ranges from 0.000 to 0.385
(within 0.07 +- 0.007 in 1 of 32). The repository holds no code for this analysis.

## Sessions at basic task proficiency (S1)

The basic (training protocol) sessions of the P1 day and the two training days before it, per
reading of P1. One value per mouse from the pooled trials.

In [ ]:
def s1_values(r):
    t = ib.window(Tm, P1[r].p1_tday0, -2, 0, ["training"])
    fits = ib.mouse_fits(t)
    g = t.groupby("subject_id")
    ses = Sc[Sc.session_id.isin(t.session_id)]
    v = pd.DataFrame({
        "perf_easy": g.apply(ib.perf_easy, include_groups=False),
        "rt": g.rt.median(),
        "n_trials": ses.groupby("subject_id").n_trials.mean(),
        "threshold": fits.threshold, "bias": fits.bias,
        "trials_per_min": (ses.n_trials / ses.duration_min).groupby(ses.subject_id).mean(),
    })
    v["lab"] = lab[v.index]
    return v, t

S1 = {}
S1_trials = {}
for r in READINGS:
    S1[r], S1_trials[r] = s1_values(r)
    print(RN[r], "mice", len(S1[r]), "| sessions per mouse", S1_trials[r].groupby("subject_id").session_id.nunique()
          .value_counts().to_dict(), "| no valid trial duration", int(S1[r].rt.isna().sum()))

## fig3c-easy-errors-basic

> At basic task proficiency, errors on easy (50% and 100%) contrasts were low, 9.5 +- 3.6%, with
> no significant difference across laboratories.

In [ ]:
rows = []
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    e = 100 - v.perf_easy
    rows.append(dict(reading=RN[r], n=len(e), mean=e.mean(), sd=e.std(), min=e.min(), max=e.max(),
                     mean_ok=abs(e.mean() - 9.5) <= 0.95, sd_ok=abs(e.std() - 3.6) <= 0.36))
print(pd.DataFrame(rows).round(2).to_string())
v = S1[READINGS[0]]
print("by lab (1a/each):", (100 - v.perf_easy).groupby(v.lab).mean().round(2).to_dict())

**Result: reproduced.** The error rate on easy trials at proficiency is 8.56 +- 3.77% (1a/each,
75 mice) and 9.77 +- 3.80% (1a/pooled, 84 mice), both within tolerance of 9.5 +- 3.6%. Under the
1a/1b readings it is 4.81 and 5.72%, as the 1b criteria require more than 90% correct. The lab
comparison is the next claim.

## fig3de-threshold-bias-basic

> At basic task proficiency the contrast threshold was similar across laboratories at
> 14.3 +- 3.8 (s.d., n = 7 laboratories) and the response bias was small at 0.3 +- 8.4.

In [ ]:
EXP_DE = [(14.3, 1.43), (0.3, 3.2), (3.8, 0.38), (8.4, 0.84)]
rows = []
for r in READINGS:
    v = S1[r]
    for how in ("over mice", "over lab means"):
        if how == "over mice":
            th, bi = v.threshold.dropna(), v.bias.dropna()
        else:
            th, bi = v.threshold.groupby(v.lab).mean(), v.bias.groupby(v.lab).mean()
        x = [th.mean(), bi.mean(), th.std(), bi.std()]
        rows.append(dict(reading=RN[r], how=how, n=len(th), thr=x[0], bias=x[1], thr_sd=x[2], bias_sd=x[3],
                         n_in_tol=sum(abs(a - e) <= t for a, (e, t) in zip(x, EXP_DE)),
                         means_ok=abs(x[0] - 14.3) <= 1.43 and abs(x[1] - 0.3) <= 3.2, all_ok=check(x, EXP_DE)))
print(pd.DataFrame(rows).round(2).to_string())
print("fits without a value:", {RN[r]: int(S1[r].threshold.isna().sum()) for r in READINGS})

**Result: reproduced.** Over mice, the threshold is 13.91 +- 3.60 and the bias -0.61 +- 8.20
(1a/each), and 13.89 +- 3.88 and 0.19 +- 9.06 (1a/pooled): all four values within tolerance under
both. Over the seven lab means the means match (13.78 to 14.87) but the s.d.s are 0.6 to 2.1,
far below 3.8 and 8.4, so the paper's s.d.s are over mice.

## fig3cde-no-lab-difference-medians and fig3cde-no-lab-difference-variances

> At basic task proficiency, easy-trial performance, contrast threshold and bias did not differ
> in median across laboratories.

> At basic task proficiency, easy-trial performance, contrast threshold and bias did not differ
> in variance across laboratories.

Kruskal-Wallis (and Levene) across labs on six variables, Benjamini-Hochberg over the six.

In [ ]:
VARS = ["perf_easy", "rt", "n_trials", "threshold", "bias", "trials_per_min"]
labtests = {}
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    out = {}
    for test in ("kruskal", "levene", "code"):
        p = np.array([ib.lab_test(v[x], v.lab, test) for x in VARS])
        out[test] = pd.Series(p, index=VARS)
        out[test + "_fdr"] = pd.Series(ib.fdr(p), index=VARS)
    labtests[r] = pd.DataFrame(out)
    print(RN[r], "n =", len(v), "| mice per lab", v.lab.value_counts().sort_index().to_dict())
    print(labtests[r].round(4).to_string(), "\n")

In [ ]:
for r in READINGS:
    lt = labtests[r]
    m = lt.loc[["perf_easy", "threshold", "bias"]]
    print(f"{RN[r]}: medians, all three FDR p >= 0.05: {bool((m.kruskal_fdr >= 0.05).all())} "
          f"(min {m.kruskal_fdr.min():.3f}); code switch: {bool((m.code_fdr >= 0.05).all())} | "
          f"variances, all three FDR p >= 0.05: {bool((m.levene_fdr >= 0.05).all())} (min {m.levene_fdr.min():.3f})")
    v = S1[r]
    eff = {x: f"{v[x].groupby(v.lab).median().min():.2f}..{v[x].groupby(v.lab).median().max():.2f}"
           for x in ["perf_easy", "threshold", "bias"]}
    sds = {x: f"{v[x].groupby(v.lab).std().min():.2f}..{v[x].groupby(v.lab).std().max():.2f}"
           for x in ["perf_easy", "threshold", "bias"]}
    print("   lab medians:", eff, "| lab s.d.:", sds)

**Result: blocked (methods_underspecified).** The corrected Kruskal-Wallis p values for
performance, threshold and bias are all above 0.05 under 1a/each (smallest 0.295), 1a/pooled
(0.265) and 1a1b/each (0.125), but under 1a1b/pooled performance differs across labs (corrected
p = 0.047; lab medians 91.1 to 97.4%). The claim's two P1 readings give different outcomes under
one reading of the open question. The paper's p values are 0.63, 0.81 and 0.81.

**Result: reproduced.** The corrected Levene p values for performance, threshold and bias are all
above 0.05 under every reading (smallest 0.223, 0.264, 0.535 and 0.705).

## fig3s2a-trial-duration

> At basic task proficiency the median trial duration (stimulus onset to feedback) was
> 468 +- 221 ms, with some differences across laboratories.

In [ ]:
rows = []
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    rt = 1000 * v.rt
    pk = labtests[r].loc["rt", "kruskal"]
    pf = labtests[r].loc["rt", "kruskal_fdr"]
    rows.append(dict(reading=RN[r], n=len(rt), median=rt.median(), sd=rt.std(), mean=rt.mean(),
                     p_kw=pk, p_kw_fdr=pf,
                     ok_uncorr=abs(rt.median() - 468) <= 46.8 and abs(rt.std() - 221) <= 22.1 and pk < 0.05,
                     ok_fdr=abs(rt.median() - 468) <= 46.8 and abs(rt.std() - 221) <= 22.1 and pf < 0.05))
print(pd.DataFrame(rows).round(4).to_string())
v = S1[READINGS[0]]
print("lab medians (ms, 1a/each):", (1000 * v.rt).groupby(v.lab).median().round(0).to_dict())

**Result: not reproduced.** The median over mice of the median trial duration is 417.5 ms
(1a/each) and 437.3 ms (1a/pooled; within 468 +- 46.8), with s.d. 225 and 227 ms (within 221 +-
22.1), but the Kruskal-Wallis test does not find the difference across labs (p = 0.085 to 0.51
uncorrected), which the criterion requires (p < 0.05). Lab 4 is slower than the others (median
702 ms against 370 to 432 ms, 1a/each).

## fig3s2b-trials-per-session

> At basic task proficiency mice performed 719 +- 223 trials per session; the difference across
> laboratories was significant but only in one laboratory relative to the rest.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
rows = []
for r in READINGS:
    v = S1[r][S1[r].rt.notna()]
    p = ib.lab_test(v.n_trials, v.lab, "anova")
    tk = pairwise_tukeyhsd(v.n_trials, v.lab)
    tab = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
    sig = tab[tab.reject]
    labs_inv = pd.concat([sig.group1, sig.group2]).value_counts()
    common = [l for l in labs_inv.index if all((sig.group1 == l) | (sig.group2 == l))] if len(sig) else []
    rows.append(dict(reading=RN[r], n=len(v), mean=v.n_trials.mean(), sd=v.n_trials.std(), p_anova=p,
                     p_anova_fdr=labtests[r].loc["n_trials", "kruskal_fdr"],
                     sig_pairs=", ".join(f"{a}-{b}" for a, b in zip(sig.group1, sig.group2)),
                     one_lab=common[0] if common else "",
                     values_ok=abs(v.n_trials.mean() - 719) <= 71.9 and abs(v.n_trials.std() - 223) <= 22.3))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))
v = S1[READINGS[0]]
print("lab means (1a/each):", v.n_trials.groupby(v.lab).mean().round(0).to_dict())

**Result: partially reproduced.** Mice performed 690 to 737 trials per session (within 719 +-
71.9), with an s.d. of 192 to 199 trials, below 223 +- 22.3. The one-way ANOVA across labs gives p
= 4.9e-05 to 0.0056, below 0.05 but not below 1e-6. The significant Tukey pairs all involve Lab 7
(412 trials per session against 614 to 886, 1a/each) under the pooled readings, and also one pair
without it under the each readings.

## Lab classifiers (fig3f-classifier-chance-basic, fig3f-positive-control, fig3s3-other-classifiers, fig4j-classifier-chance-full)

The classifier draws 8 mice from each of the seven labs (Methods; claims file header). Mice per
lab available under each selection:

In [ ]:
counts = pd.DataFrame({RN[r]: S1[r].lab.value_counts() for r in READINGS}).reindex(ib.LABS).fillna(0).astype(int)
counts["S2 (P2)"] = lab[P2.index].value_counts().reindex(ib.LABS).fillna(0).astype(int)
counts["complete history"] = lab[complete].value_counts().reindex(ib.LABS).fillna(0).astype(int)
print(counts.to_string())
print("labs with fewer than 8 mice:", {c: list(counts.index[counts[c] < 8]) for c in counts.columns})
# Not deciding any outcome: the Naive Bayes classifier with as many mice per lab as the smallest lab
# holds, to show what the data give. First, the direct leave-one-out GaussianNB against scikit-learn.
from sklearn.naive_bayes import GaussianNB
rng_nb = np.random.default_rng(7)
Xt = rng_nb.normal(size=(35, 3)); yt = np.repeat(np.arange(7), 5)
agree = np.mean(ib.loo_gaussian_nb(Xt, yt) == cross_val_predict(GaussianNB(), Xt, yt, cv=LeaveOneOut()))
print("\nleave-one-out GaussianNB, direct against scikit-learn, agreement on synthetic data:", agree)
for r in READINGS[:1]:
    v = S1[r][S1[r].rt.notna()].dropna(subset=["threshold", "bias"])
    k = int(v.lab.value_counts().min())
    acc, null = ib.classify_labs(v[["perf_easy", "threshold", "bias"]].to_numpy(), v.lab.to_numpy(), "nb",
                                 n_iter=2000, per_lab=k)
    print(f"variant ({RN[r]}, {k} mice per lab): accuracy {acc.mean():.3f} +- {acc.std():.3f}, "
          f"2.5th percentile {np.percentile(acc, 2.5):.3f}, null mean {null.mean():.3f}")

**Result: blocked (other).** The classifier draws 8 mice from each lab, and at basic proficiency
at least two labs hold fewer than 8 mice under every reading (Lab 4 and Lab 7 hold 7 and 6 mice
under 1a/pooled; Lab 7 holds 7 mice with their whole training in the dataset). As a variant that
decides nothing, the Naive Bayes classifier with 6 mice per lab (1a/each) has accuracy 0.157 +-
0.052 against a null of 0.113, and its 2.5th percentile (0.071) is below the null mean.

**Result: blocked (other).** As for fig3f-classifier-chance-basic, on which it depends: fewer
than 8 mice in some labs under every reading.

**Result: blocked (other).** As for fig3f-classifier-chance-basic, on which it depends: fewer
than 8 mice in some labs under every reading.

**Result: blocked (other).** At full task proficiency five labs hold fewer than 8 mice (Lab 2,
4, 5, 6 and 7 hold 7, 5, 7, 6 and 5).

## lab-link

The lab numbers (Lab 1 to 7) and, for the positive control, the labs' time zones come from the
paper's code by institution (claims file header). The dataset names each session's institution.

In [ ]:
inst = S.groupby("institution").agg(lab_number=("lab_number", "first"), mice=("subject_id", "nunique"),
                                    labs_in_files=("lab", lambda x: sorted(set(x))))
inst["time_zone"] = inst.lab_number.map(ib.TIME_ZONE)
print(inst.to_string())
print("institutions matched to a lab number:", int(inst.lab_number.notna().sum()), "of", len(inst),
      "| distinct lab numbers:", inst.lab_number.nunique())
h = pd.to_datetime(S.start.str[:19]).dt.hour + pd.to_datetime(S.start.str[:19]).dt.minute / 60
med = h.groupby(S.lab_number).median()
print("median start hour by lab (stated as UTC):", med.round(1).to_dict())
print("spread of lab medians %.1f h; if the times were UTC, Lab 7 (UTC-7) would start about 7 h later than Lab 1"
      % (med.max() - med.min()))

**Result: reproduced.** The seven institutions in the files match the code's seven labs one to
one (7 of 7), with CSHL holding two lab groups (churchlandlab and zadorlab) and SWC two (hoferlab
and mrsicflogellab). The time zones cannot be confirmed from the data: `session_start_time` is
stated in UTC, but the median start hours of all labs fall between 11.0 and 14.4 h, Lab 7 at 11.0
h like Lab 1, so the times are local clock times labelled UTC.

## fig3s1a-choice-consistency-basic

> In the basic task, mouse choices were no more consistent within laboratories than across
> laboratories.

Fraction of rightward choices per signed contrast (50% contrast trials dropped), per mouse; 10,000
shuffles of lab labels. Main: S1 sessions; variant (code): the P1 day and the two days after.

In [ ]:
def choice_matrix(t, by_block=False):
    t = t[(t.abs_contrast != 50) & t.choice_right.notna()]
    cols = ["signed_contrast"] + (["probabilityLeft"] if by_block else [])
    return t.groupby(["subject_id"] + cols).choice_right.mean().unstack(cols)

rows = []
for r in READINGS:
    for win in ("S1", "code [0, 2]"):
        t = S1_trials[r] if win == "S1" else ib.window(Tm, P1[r].p1_tday0, 0, 2, ["training", "biased"])
        if win != "S1":  # basic-task trials, and the 50:50 trials of full-task sessions
            t = t[(t.task == "training") | np.isclose(t.probabilityLeft, 0.5)]
        M = choice_matrix(t)
        obs, null, p = ib.consistency_test(M, lab[M.index], 10000, seed=11)
        rows.append(dict(reading=RN[r], sessions=win, mice=len(M), consistency=obs,
                         null_mean=null.mean(), p=p, ok=p >= 0.05))
print(pd.DataFrame(rows).round(4).to_string())

**Result: reproduced.** Choices are not more consistent within labs than in shuffles: p = 0.287
to 0.645 on S1 under every reading, and 0.156 to 0.641 on the code's sessions.

## Sessions at full task proficiency (S2)

The full task (biased protocol) sessions of the P2 day and the two training days before it.

In [ ]:
S2t = ib.window(Tm, P2.p2_tday0, -2, 0, ["biased"])
print("P2 mice", len(P2), "| sessions per mouse", S2t.groupby("subject_id").session_id.nunique().value_counts().to_dict(),
      "| trials", len(S2t), "| block values", S2t.probabilityLeft.round(2).value_counts().to_dict())
blk = {"2080": 0.2, "8020": 0.8}
S2fits = ib.mouse_fits(S2t.assign(block=np.select([np.isclose(S2t.probabilityLeft, 0.2),
                                                    np.isclose(S2t.probabilityLeft, 0.8)],
                                                   ["2080", "8020"], "5050")),
                       by=("subject_id", "block")).unstack("block")
print("fits without a value:", int(S2fits.isna().any(axis=1).sum()))

## fig3s1bc-choice-consistency-full

> In the full task, neither choice fractions nor the bias shift between blocks were more
> consistent within laboratories than expected by chance.

In [ ]:
Mb = choice_matrix(S2t, by_block=True)
obs_b, null_b, p_b = ib.consistency_test(Mb, lab[Mb.index], 10000, seed=12)
Mfrac = choice_matrix(S2t, by_block=True)
shift = Mfrac.xs(0.2, axis=1, level="probabilityLeft") - Mfrac.xs(0.8, axis=1, level="probabilityLeft")
obs_c, null_c, p_c = ib.consistency_test(shift, lab[shift.index], 10000, seed=13)
print(f"(b) choices per contrast and block: mice {len(Mb)}, consistency {obs_b:.3f}, null mean {null_b.mean():.3f}, p = {p_b:.4f}")
print(f"(c) bias shift per contrast: mice {len(shift)}, consistency {obs_c:.3f}, null mean {null_c.mean():.3f}, p = {p_c:.4f}")
print("both p >= 0.05:", p_b >= 0.05 and p_c >= 0.05)

**Result: not reproduced.** In the full task, choices per contrast and block are more consistent
within labs than in shuffles (consistency 98.2 against a null mean of 82.5, p = 0.017, 60 mice).
The bias shift is not (p = 0.154), as in the paper.

## fig3s4-looser-criterion

> With a looser proficiency criterion (80% correct on easy trials over three sessions, n = 150
> mice), behavior was again consistent across laboratories; only the number of trials per session
> differed slightly, and a classifier failed to identify a mouse's laboratory.

For each mouse, basic-task sessions with five or more signed contrasts; the first three
consecutive such sessions each above 80% correct on easy trials.

In [ ]:
ses_tr = Sc[Sc.task == "training"].sort_values("start")
st = Tm[Tm.task == "training"].groupby("session_id").agg(perf=("correct", lambda x: np.nan), n_signed=("signed_contrast", "nunique"))
st["perf"] = Tm[Tm.task == "training"].groupby("session_id").apply(ib.perf_easy, include_groups=False)
ses_tr = ses_tr.join(st, on="session_id")
ses_tr = ses_tr[ses_tr.n_signed >= 5]
loose = {}
for sid, d in ses_tr.groupby("subject_id"):
    ok = (d.perf > 80).to_numpy()
    for j in range(2, len(d)):
        if ok[j - 2: j + 1].all():
            loose[sid] = d.session_id.iloc[j - 2: j + 1].tolist()
            break
print("mice meeting the looser criterion:", len(loose), "| within 150 +- 15:", abs(len(loose) - 150) <= 15)
print("by lab:", lab[list(loose)].value_counts().sort_index().to_dict())
tl = Tm[Tm.session_id.isin({x for v in loose.values() for x in v})]
fl = ib.mouse_fits(tl)
g = tl.groupby("subject_id")
sl = Sc[Sc.session_id.isin(tl.session_id)]
vl = pd.DataFrame({"perf_easy": g.apply(ib.perf_easy, include_groups=False), "rt": g.rt.median(),
                   "n_trials": sl.groupby("subject_id").n_trials.mean(), "threshold": fl.threshold, "bias": fl.bias,
                   "trials_per_min": (sl.n_trials / sl.duration_min).groupby(sl.subject_id).mean()})
vl["lab"] = lab[vl.index]
p = np.array([ib.lab_test(vl[x], vl.lab, "kruskal") for x in VARS])
pc = np.array([ib.lab_test(vl[x], vl.lab, "code") for x in VARS])
print(pd.DataFrame({"kruskal": p, "kruskal_fdr": ib.fdr(p), "code_switch_fdr": ib.fdr(pc)}, index=VARS).round(4))
pf = pd.Series(ib.fdr(p), index=VARS)
print("trials per session corrected p < 0.05:", pf["n_trials"] < 0.05,
      "| other five >= 0.05:", bool((pf.drop("n_trials") >= 0.05).all()))

**Result: not reproduced.** 91 mice meet the looser criterion, against 150 +- 15. Trials per
session differ across labs (corrected p = 0.0022), as in the paper, but so does the number of
trials per minute (corrected p = 0.0022), where the paper found no difference. The classifier part
was not run: Lab 4 and Lab 7 hold 7 mice each.

## First full-task sessions (P1b)

The code's P1b is the first full-task session with status trained_1b. Without a status in the
files, P1b is taken here as the first full-task (biased protocol) day on or after P1, per reading.

In [ ]:
def p1b_day(r):
    p = P1[r].p1_tday0
    fb = Dc[Dc.tasks.apply(lambda t: "biased" in t) & Dc.subject_id.isin(p.index)]
    fb = fb[fb.tday0 >= fb.subject_id.map(p)]
    return fb.groupby("subject_id").tday0.min()

P1b = {r: p1b_day(r) for r in READINGS}
print({RN[r]: len(P1b[r]) for r in READINGS})
print("days from P1 to P1b (1a/each):", (P1b[READINGS[0]] - P1[READINGS[0]].p1_tday0.reindex(P1b[READINGS[0]].index))
      .describe()[["min", "50%", "max"]].to_dict())

## fig3s5-first-full-sessions

> In the first three sessions of the full task, bias in each block, contrast threshold in each
> block and performance on 100% contrast trials did not vary significantly across laboratories,
> and a classifier could not predict lab membership.

In [ ]:
def first_full(r, how):
    if how == "legend: first three biased days":
        t = Tm[Tm.subject_id.isin(P1b[r].index) & (Tm.task == "biased")]
        t = t[t.tday0 >= t.subject_id.map(P1b[r])]
        firstdays = t.groupby("subject_id").tday0.apply(lambda x: sorted(x.unique())[:3])
        return t[[d in firstdays[s] for s, d in zip(t.subject_id, t.tday0)]]
    return ib.window(Tm, P1b[r], 1, 3, ["biased"])

ff = {}
rows = []
for r in READINGS:
    for how in ("legend: first three biased days", "code: days P1b+1 to P1b+3"):
        t = first_full(r, how)
        t = t.assign(block=np.select([np.isclose(t.probabilityLeft, 0.2), np.isclose(t.probabilityLeft, 0.8)],
                                     ["2080", "8020"], "5050"))
        fits = ib.mouse_fits(t, by=("subject_id", "block")).unstack("block")
        v = pd.DataFrame({f"{p}_{b}": fits[(p, b)] for p in ("bias", "threshold") for b in ("2080", "5050", "8020")})
        v["perf_100"] = t[t.abs_contrast == 100].groupby("subject_id").correct.mean() * 100
        v["perf_easy_code"] = t[t.abs_contrast >= 50].groupby("subject_id").correct.mean() * 100
        v["lab"] = lab[v.index]
        ff[(r, how)] = v
        ps = {c: ib.lab_test(v[c], v.lab, "kruskal") for c in v.columns if c not in ("lab", "perf_easy_code")}
        from statsmodels.stats.multitest import multipletests
        holm = multipletests(list(ps.values()), method="holm-sidak")[1]
        rows.append(dict(reading=RN[r], sessions=how, mice=len(v), **{k: round(x, 3) for k, x in ps.items()},
                         p_perf_easy_code=round(ib.lab_test(v.perf_easy_code, v.lab, "kruskal"), 3),
                         min_p=min(ps.values()), min_p_holm=holm.min(), all_ns=min(ps.values()) >= 0.05))
print(pd.DataFrame(rows).to_string())

**Result: not reproduced.** Under the legend's sessions (the first three full-task days) at least
one of the seven Kruskal-Wallis p values is below 0.05 under every reading (smallest 0.017 to
0.033); under 1a/each, threshold in 20:80 blocks p = 0.036 and performance on 100% contrast p =
0.031. With the code's Holm-Sidak correction, a variant, the smallest is 0.116 to 0.209. The code's
sessions (days P1b+1 to P1b+3) give smaller p values (0.002 to 0.013). The classifier part was not
run (fewer than 8 mice in some labs).

## fig3s5a-prior-used-from-first-sessions

> Mice incorporated the block priors into their choices from the first sessions of the full task.

In [ ]:
rows = []
for (r, how), v in ff.items():
    d = (v.bias_8020 - v.bias_2080).dropna()
    w = stats.wilcoxon(v.bias_8020, v.bias_2080, nan_policy="omit")
    rows.append(dict(reading=RN[r], sessions=how, n=len(d), median_diff=d.median(), positive=f"{int((d > 0).sum())}/{len(d)}",
                     p=w.pvalue, above_5=f"{int((d > 5).sum())}/{len(d)}", ok=d.median() > 0 and w.pvalue < 0.05))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** In the first full-task sessions, bias is higher in 80:20 than in 20:80
blocks under every reading (median difference 4.3 to 6.4; 49 of 51 to 68 of 71 mice; Wilcoxon p =
6.2e-10 or smaller). Between 25 of 51 and 31 of 68 mice exceed the proficiency threshold of 5,
a variant.

## full-task-training-time

> Mice that reached full task proficiency took 31.5 +- 16.1 days, or 20,494 +- 10,980 trials
> (s.d., n = 98), from the start of training; the fastest took 8 days (4691 trials), the slowest
> 81 days (61,316 trials).

In [ ]:
nd = P2.p2_tday0 + 1
dd = Dc[Dc.subject_id.isin(nd.index)]
ntr = dd[dd.tday0 < dd.subject_id.map(nd)].groupby("subject_id").n_trials.sum()
EXP_FT = [(31.5, 3.15), (16.1, 1.61), (20494, 2049), (10980, 1098), (8, 0.8), (81, 8.1), (4691, 469), (61316, 6132)]
v = [nd.mean(), nd.std(), ntr.mean(), ntr.std(), nd.min(), nd.max(), ntr.min(), ntr.max()]
for (e, t), x, name in zip(EXP_FT, v, ["mean days", "sd days", "mean trials", "sd trials", "min days", "max days",
                                       "min trials", "max trials"]):
    print(f"{name}: {x:.1f} (paper {e}, within tolerance: {abs(x - e) <= t})")
print("mice:", len(nd), "| by lab:", lab[nd.index].value_counts().sort_index().to_dict())
active = Dc.groupby("subject_id").date.max() >= "2020-03-16"
print("last date of the dataset", Dc.date.max(), "| whole-training mice without P2 with a session in its last week:",
      int((active & ~active.index.isin(P2.index)).sum()))

**Result: partially reproduced.** The 60 mice that reach full task proficiency take 29.4 +- 14.1
days and 19,637 +- 9,173 trials: the means are within tolerance, the s.d.s are not (16.1 and
10,980 in the paper). The fastest takes 10 days (5,592 trials) and the slowest 64 days (43,500
trials), against 8 (4,691) and 81 (61,316). 30 mice without P2 were still training in the
dataset's last week, which cuts off slow learners.

## fig4cd-biasshift-largest-at-zero

> Block structure had the greatest effect on choices when sensory evidence was absent; the shift in
> rightward choices between 20:80 and 80:20 blocks was highest at 0% contrast in every laboratory,
> averaging 28.5%.

In [ ]:
CON = sorted(S2t.signed_contrast.unique())
def curve(row, b, c):
    p = {k: row[(k, b)] for k in ("bias", "threshold", "lapselow", "lapsehigh")}
    return 100 * ib.gamma_curve(p, c)
sh = pd.DataFrame({c: [curve(S2fits.loc[m], "2080", c) - curve(S2fits.loc[m], "8020", c) for m in S2fits.index]
                   for c in CON}, index=S2fits.index)
print("signed contrasts:", CON)
print("mean shift over mice by contrast:", sh.mean().round(2).to_dict())
by_lab = sh.groupby(lab[sh.index]).mean()
print(by_lab.round(1).to_string())
at0 = sh[0.0]
print(f"mean shift at 0%: {at0.mean():.2f} (n = {at0.notna().sum()}), within 28.5 +- 2.85: {abs(at0.mean() - 28.5) <= 2.85}")
print("labs whose mean shift is highest at 0%:", int((by_lab.idxmax(axis=1) == 0.0).sum()), "of", len(by_lab),
      "| argmax per lab:", by_lab.idxmax(axis=1).to_dict())

**Result: reproduced.** The mean shift in fitted rightward choices between 20:80 and 80:20 blocks
is 28.00% at 0% contrast (60 mice; within 28.5 +- 2.85), and in each of the 7 labs the lab mean is
highest at 0%.

## fig4de-biasshift-no-lab-difference

> The bias shift at 0% contrast did not differ across laboratories more than within them.

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
dfa = pd.DataFrame({"shift": at0, "lab": lab[at0.index]}).dropna()
aov = sm.stats.anova_lm(smf.ols("shift ~ C(lab)", data=dfa).fit(), typ=2)
print(aov)
print("p >= 0.05:", aov["PR(>F)"].iloc[0] >= 0.05, "| lab means:", dfa.groupby("lab")["shift"].mean().round(1).to_dict())

**Result: reproduced.** The one-way ANOVA of the shift at 0% across labs gives F = 2.07, p =
0.072 (paper F = 1.345, p = 0.2455), with lab means from 23.0 to 36.2%.

## fig4f, fig4gh, fig4i: block effects on the psychometric parameters

> Block type did not significantly affect the contrast threshold.

> Block type changed lapse rates, which were higher on the left in 20:80 blocks and higher on the
> right in 80:20 blocks.

> Block type changed the overall bias consistently, with curves shifting left in 20:80 blocks and
> right in 80:20 blocks.

In [ ]:
def wil(a, b):
    d = (a - b).dropna()
    return d.median(), stats.wilcoxon(a, b, nan_policy="omit").pvalue, int((d > 0).sum()), len(d)
md_, p, npos, n = wil(S2fits[("threshold", "2080")], S2fits[("threshold", "8020")])
print(f"threshold 20:80 - 80:20: median {md_:.2f}, {npos}/{n} positive, Wilcoxon p = {p:.4g}; p >= 0.05: {p >= 0.05}")

**Result: reproduced.** The threshold does not differ between 20:80 and 80:20 blocks (median
difference 0.08, 31 of 60 positive, Wilcoxon p = 0.80; paper p = 0.85).

In [ ]:
m1, p1, n1, n = wil(S2fits[("lapselow", "2080")], S2fits[("lapselow", "8020")])
m2, p2, n2, _ = wil(S2fits[("lapsehigh", "8020")], S2fits[("lapsehigh", "2080")])
print(f"left lapse, 20:80 - 80:20: median {m1:.4f}, {n1}/{n} positive, p = {p1:.3g}")
print(f"right lapse, 80:20 - 20:80: median {m2:.4f}, {n2}/{n} positive, p = {p2:.3g}")
print("directions hold:", m1 > 0 and m2 > 0, "| p < 1e-6 and < 1e-7:", p1 < 1e-6 and p2 < 1e-7,
      "| both p < 0.05:", p1 < 0.05 and p2 < 0.05)
print("smallest p a Wilcoxon test can give with n = %d (normal approximation, all differences one sign): %.2g"
      % (n, stats.wilcoxon(np.arange(1, n + 1)).pvalue))

**Result: partially reproduced.** The left lapse is higher in 20:80 blocks (median difference
0.015, 46 of 60 mice, p = 2.3e-06) and the right lapse higher in 80:20 blocks (0.018, 45 of 60, p =
2.2e-05). The directions hold with p < 0.05, but not with p < 1e-6 and < 1e-7.

In [ ]:
m3, p3, n3, n = wil(S2fits[("bias", "8020")], S2fits[("bias", "2080")])
print(f"bias 80:20 - 20:80: median {m3:.2f}, {n3}/{n} positive, p = {p3:.3g}")
print("direction holds:", m3 > 0, "| p < 1e-16:", p3 < 1e-16, "| p < 0.05:", p3 < 0.05)

**Result: partially reproduced.** Bias is higher in 80:20 than in 20:80 blocks in all 60 mice
(median difference 7.11, p = 1.6e-11). This is the smallest p the test gives with 60 mice, so the
paper's p < 1e-16 (98 mice) cannot be reached, and the criterion gives partial.

## fig4fi-no-lab-difference

> Threshold, lapses and bias in each block type showed no significant differences across
> laboratories.

In [ ]:
v8 = pd.DataFrame({f"{p}_{b}": S2fits[(p, b)] for p in ("threshold", "lapselow", "lapsehigh", "bias")
                   for b in ("8020", "2080")})
pv = np.array([ib.lab_test(v8[c], lab[v8.index], "code") for c in v8.columns])
pk = np.array([ib.lab_test(v8[c], lab[v8.index], "kruskal") for c in v8.columns])
print(pd.DataFrame({"code_switch": pv, "code_switch_fdr": ib.fdr(pv), "kruskal_fdr": ib.fdr(pk)}, index=v8.columns).round(4))
print("all eight corrected p >= 0.05:", bool((ib.fdr(pv) >= 0.05).all()))

**Result: reproduced.** None of the eight per-block parameters differs across labs after
correction (corrected p 0.19 to 0.88).

## Probabilistic choice model (fig5cf, fig5dg, fig5s2)

Sessions per the code: training days P1b-2 to P1b+3, split by protocol into basic and full; or
(b) the last three basic days before P1b and the first three full days. Contrasts with 50%
(Methods) or without (code).

In [ ]:
def glm_trials(r, how):
    if how == "code: P1b-2..P1b+3":
        t = ib.window(Tm, P1b[r], -2, 3, ["training", "biased"])
        return t[t.task == "training"], t[t.task == "biased"]
    tb = ib.window(Tm, P1b[r], -60, -1, ["training"])
    lastdays = tb.groupby("subject_id").tday0.apply(lambda x: sorted(x.unique())[-3:])
    tb = tb[[d in lastdays[s] for s, d in zip(tb.subject_id, tb.tday0)]]
    tf = ib.window(Tm, P1b[r], 0, 60, ["biased"])
    firstdays = tf.groupby("subject_id").tday0.apply(lambda x: sorted(x.unique())[:3])
    tf = tf[[d in firstdays[s] for s, d in zip(tf.subject_id, tf.tday0)]]
    return tb, tf

GLM = {}
for r in READINGS:
    for how in ("code: P1b-2..P1b+3", "(b) last 3 basic, first 3 full"):
        tb, tf = glm_trials(r, how)
        for drop50 in (False, True):
            items, keys = [], []
            for task, t in (("basic", tb), ("full", tf)):
                for sid, d in t.groupby("subject_id"):
                    items.append((ib.glm_design(d, task == "full", drop50), task == "full", drop50))
                    keys.append((sid, task))
            res = ib.fit_glms(items)
            df = pd.DataFrame(res, index=pd.MultiIndex.from_tuples(keys, names=["subject_id", "task"]))
            GLM[(r, how, drop50)] = df
print("fits:", {f"{RN[k[0]]} | {k[1]} | drop50={k[2]}": f"{len(v)} ({int(v.get('error', pd.Series(dtype=object)).notna().sum())} failed)"
                for k, v in GLM.items()})

## fig5cf-glm-weights

> In a logistic choice model fitted to each mouse, visual weights grew with contrast to above 3 at
> 100% contrast, while weights for past choices were much smaller and positive after both rewarded
> (basic 0.19, full 0.42) and unrewarded (basic 0.33, full 0.46) trials, indicating perseverance.

In [ ]:
EXP_W = [(0.19, 0.019), (0.42, 0.042), (0.33, 0.033), (0.46, 0.046)]
rows = []
for (r, how, drop50), df in GLM.items():
    m = df.groupby(level="task").mean(numeric_only=True)
    cols = [c for c in ["c6_25", "c12_5", "c25_0", "c50_0", "c100_0"] if c in m.columns]
    inc = all(np.all(np.diff(m.loc[t, cols].dropna().to_numpy()) > 0) for t in ("basic", "full"))
    w = [m.loc["basic", "rewarded"], m.loc["full", "rewarded"], m.loc["basic", "unrewarded"], m.loc["full", "unrewarded"]]
    rows.append(dict(reading=RN[r], sessions=how, drop50=drop50,
                     n=f"{(df.index.get_level_values('task') == 'basic').sum()},{(df.index.get_level_values('task') == 'full').sum()}",
                     with_50=int(df.get("c50_0", pd.Series(dtype=float)).notna().sum()),
                     w100_basic=m.loc["basic", "c100_0"], w100_full=m.loc["full", "c100_0"], increasing=inc,
                     rew_basic=w[0], rew_full=w[1], unrew_basic=w[2], unrew_full=w[3],
                     n_hist_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(w, EXP_W)),
                     ok=m.loc["basic", "c100_0"] > 3 and m.loc["full", "c100_0"] > 3 and inc and check(w, EXP_W)))
glm_tab = pd.DataFrame(rows)
print(glm_tab.round(3).to_string())

**Result: partially reproduced.** The 100% contrast weight is above 3 in both tasks (3.51 to
4.13), the visual weights increase with contrast without 50% trials, and the past-choice weights
are positive. Three of the four history weights are within tolerance under most readings (code
sessions, without 50%, 1a/each: rewarded 0.209 and 0.433, unrewarded 0.273 and 0.435, against
0.19, 0.42, 0.33 and 0.46), but no reading has all four. The unrewarded weight in the basic task is
0.256 to 0.335 and within 0.33 +- 0.033 only where the full-task unrewarded weight is not.

## fig5dg-history-weights-higher-full

> Previous choices influenced behavior more strongly in the full task than in the basic task,
> after both rewarded and unrewarded trials.

In [ ]:
rows = []
for (r, how, drop50), df in GLM.items():
    w = df[["rewarded", "unrewarded"]].unstack("task").dropna()
    tr = stats.ttest_rel(w[("rewarded", "full")], w[("rewarded", "basic")])
    tu = stats.ttest_rel(w[("unrewarded", "full")], w[("unrewarded", "basic")])
    rows.append(dict(reading=RN[r], sessions=how, drop50=drop50, n=len(w), t_rew=tr.statistic, p_rew=tr.pvalue,
                     t_unrew=tu.statistic, p_unrew=tu.pvalue,
                     ok=tr.statistic > 0 and tu.statistic > 0 and tr.pvalue < 0.05 and tu.pvalue < 0.05))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** On the code's sessions, both history weights are larger in the full task
under every reading (rewarded t = 8.85 to 10.79; unrewarded t = 3.55 to 4.07; all p < 0.001). On
the sessions of reading (b) the unrewarded difference is not significant (p = 0.059 to 0.34).

## fig5s2-glm-accuracy

> The model predicted individual mouse choices well, with cross-validated accuracy of
> 81.03 +- 5.1% in the basic task and 82.1 +- 5.8% in the full task, similar across
> laboratories; mean condition numbers were 2.4 and 3.2.

In [ ]:
EXP_A = [(81.03, 8.1), (82.1, 8.2), (2.4, 0.24), (3.2, 0.32)]
rows = []
for (r, how, drop50), df in GLM.items():
    m = df.groupby(level="task").mean(numeric_only=True)
    s = df.groupby(level="task").std(numeric_only=True)
    for acc in ("acc_threshold", "acc_prob"):
        v = [m.loc["basic", acc], m.loc["full", acc], m.loc["basic", "cond"], m.loc["full", "cond"]]
        rows.append(dict(reading=RN[r], sessions=how, drop50=drop50, accuracy=acc,
                         basic=f"{v[0]:.1f} +- {s.loc['basic', acc]:.1f}", full=f"{v[1]:.1f} +- {s.loc['full', acc]:.1f}",
                         cond_basic=v[2], cond_full=v[3],
                         n_in_tol=sum(abs(x - e) <= t for x, (e, t) in zip(v, EXP_A)), ok=check(v, EXP_A)))
print(pd.DataFrame(rows).round(2).to_string())
# spread over fold seeds, one reading
r0 = (READINGS[0], "code: P1b-2..P1b+3", True)
tb, tf = glm_trials(READINGS[0], "code: P1b-2..P1b+3")
seed_acc = []
for seed in range(5):
    items = [(ib.glm_design(d, False, True), False, True, 5, seed) for _, d in tb.groupby("subject_id")]
    seed_acc.append(np.mean([x["acc_threshold"] for x in ib.fit_glms(items) if "error" not in x]))
print("basic-task accuracy (1a/each, code sessions, no 50%) over fold seeds 0-4:", np.round(seed_acc, 2))

**Result: reproduced.** Without 50% trials, the cross-validated accuracy (legend reading) is 81.8
to 82.7% in the basic task and 83.4 to 84.0% in the full task, and the condition numbers are 2.36
to 2.39 and 3.14 to 3.15: all four within tolerance under every reading. The code's reading of
accuracy (mean probability of the actual choice) gives 74.0 to 77.1%, also within tolerance.
Accuracy varies by less than 0.1% over fold seeds.

## fig5s2-weights-similar-across-labs

> Fitted model weights and predictive accuracy were similar across laboratories.

In [ ]:
rows = []
for (r, how, drop50), df in GLM.items():
    if not drop50:
        continue
    ps = {}
    for task in ("basic", "full"):
        d = df.xs(task, level="task")
        cols = ["c6_25", "c12_5", "c25_0", "c100_0", "rewarded", "unrewarded", "Intercept"] + (["block"] if task == "full" else []) \
            + ["pseudo_r2", "acc_threshold"]
        for c in cols:
            ps[f"{task}:{c}"] = ib.lab_test(d[c], lab[d.index], "code")
    ps = pd.Series(ps)
    rows.append(dict(reading=RN[r], sessions=how, tests=len(ps), n_sig=int((ps < 0.05).sum()),
                     sig=", ".join(f"{k} {v:.3f}" for k, v in ps[ps < 0.05].items()), min_p=ps.min()))
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))

**Result: not reproduced.** Of the 19 lab tests, 3 to 7 give p < 0.05 under every reading,
mostly the visual weights (6.25% contrast in both tasks, p = 0.003 to 0.044), where the criterion
allows at most two for partial.

## fig5s1b-win-stay-lose-switch

> After correcting for slow fluctuations in decision bound, most mice used a win-stay lose-switch
> strategy, in both the basic and full tasks.

In [ ]:
def history_shift(t, cond_prefix, extra=None):
    """Fitted rightward % at 0% after a right minus after a left choice (of the previous or next
    trial), separately after rewarded and unrewarded trials; fits with more than 50 trials."""
    out = {}
    pairs, keys = [], []
    for (sid, rew, side), d in t.groupby(["subject_id", f"{cond_prefix}_reward", f"{cond_prefix}_choice"]):
        if side == 0 or len(d) <= 50:
            continue
        pairs.append((d.signed_contrast.to_numpy(), d.choice_right.to_numpy()))
        keys.append((sid, rew, side))
    fits = ib.fit_many(pairs)
    f = pd.DataFrame(fits, index=pd.MultiIndex.from_tuples(keys, names=["subject_id", "rew", "side"]))
    p0 = f.apply(lambda row: 100 * ib.gamma_curve(row, 0.0) if row.notna().all() else np.nan, axis=1)
    p0 = p0.unstack("side")
    return (p0.get(1.0) - p0.get(-1.0)).unstack("rew")

HIST = {}
def strategy(r, by_contrast=False):
    if r not in HIST:
        tt = ib.history_columns(ib.window(Tm, P1b[r], -2, 3, ["training", "biased"]))
        HIST[r] = tt[tt.choice_right.notna()]
    tt = HIST[r]
    res = {}
    for task, t in (("basic", tt[tt.task == "training"]), ("full", tt[tt.task == "biased"])):
        n_by_c = t.signed_contrast.value_counts()
        t = t[t.signed_contrast.isin(n_by_c[n_by_c >= 100].index)]
        if not by_contrast:
            res[task] = history_shift(t, "prev") - history_shift(t, "next")
        else:
            out = {}
            for name, cs in (("0", [0.0]), ("6.25", [6.25]), ("0+6.25", [0.0, 6.25]), ("100", [100.0])):
                tp = t[t.prev_contrast.isin(cs) & (t.prev_reward == 1)]
                tn = t[t.next_contrast.isin(cs) & (t.next_reward == 1)]
                out[name] = (history_shift(tp, "prev") - history_shift(tn, "next")).get(1.0)
            res[task] = pd.DataFrame(out)
    return res

rows = []
STRAT = {}
for r in READINGS:
    STRAT[r] = strategy(r)
    for task, c in STRAT[r].items():
        c = c.dropna()
        win, lose = c[1.0], c[0.0]
        rows.append(dict(reading=RN[r], task=task, mice=len(c), post_reward_median=win.median(), post_error_median=lose.median(),
                         frac_win_stay=(win > 0).mean(), frac_lose_switch=(lose < 0).mean(),
                         frac_both=((win > 0) & (lose < 0)).mean()))
ws = pd.DataFrame(rows)
print(ws.round(3).to_string())
print("\nmore than half with both, in each task, per reading:",
      ws.groupby("reading").frac_both.apply(lambda x: bool((x > 0.5).all())).to_dict())

**Result: reproduced.** After correction, more than half the mice both stay after a reward and
switch after an error, in each task and under every reading (basic 0.57 to 0.62; full 0.84 to
0.88). The median corrected shift is 4.8 to 5.7% after rewards and -8.2 to -9.1% after errors in
the basic task.

## fig5s1c-updating-by-contrast

> After rewarded trials, choice updating was largest when the previous stimulus had low contrast
> and much smaller after high-contrast rewarded trials.

In [ ]:
rows = []
for r in READINGS:
    res = strategy(r, by_contrast=True)
    for task, d in res.items():
        m = d.mean()
        rows.append(dict(reading=RN[r], task=task, **{f"n_{k}": int(d[k].notna().sum()) for k in d.columns},
                         **{f"mean_{k}": m[k] for k in d.columns},
                         each_above_100=m["0"] > m["100"] and m["6.25"] > m["100"], pooled_above_100=m["0+6.25"] > m["100"]))
uc = pd.DataFrame(rows)
print(uc.round(2).to_string())
print("\nper reading, both tasks:", uc.groupby("reading")[["each_above_100", "pooled_above_100"]].all().to_dict())

**Result: reproduced.** After rewarded trials the mean corrected updating after 0% and after
6.25% contrast exceeds that after 100% in both tasks under every reading (1a/each, full task: 12.20
and 8.61 against 6.10%). In the basic task the 0% value rests on 3 to 4 mice, as few 0% trials
pass the 50-trial rule; with 0% and 6.25% pooled the result is the same.

## session-end-criteria

> The session-ending criterion was not mandatory; 5,316 of 10,903 sessions ended earlier and
> 5,587 later, the latter continuing for a median of 19 +- 16 extra trials (m.a.d.).

End status from the rig settings (`SUBJECT_DISENGAGED_TRIGGERED`, `_TRIALNUM`); for sessions
without one, recomputed from the three criteria of the Figure 1, figure supplement 2 legend.

In [ ]:
def end_table(sel):
    s = S[S.subject_id.isin(sel)].copy()
    recompute = s.disengaged.isna()
    trig = s.disengaged_trial.where(s.disengaged.isin([1, 2, 3]))
    tt = T[T.session_id.isin(s.session_id[recompute])]
    rc = {}
    for sid, d in tt.groupby("session_id"):
        d = d.sort_values("start_time")
        rc[sid] = ib.end_criterion_trial(d.stimOn_times.to_numpy(), d.rt.to_numpy(), d.start_time.iloc[0])
    s["trigger"] = trig
    s.loc[recompute, "trigger"] = s.loc[recompute, "session_id"].map(rc)
    s["met"] = s.trigger.notna()
    s["extra"] = s.n_trials - s.trigger
    return s, int(recompute.sum())

for name, sel in (("all project mice", mice), ("whole-training mice", complete)):
    s, nre = end_table(sel)
    day = s.groupby(["subject_id", "date"]).agg(met=("met", "any"), dur=("duration_min", "sum"))
    day_met = day.met | (day.dur >= 90)
    ex = s.extra[s.met]
    print(f"{name}: sessions {len(s)} (recomputed {nre}), met an end criterion {int(s.met.sum())}, ended earlier {int((~s.met).sum())}")
    print(f"   training days {len(day)}, met (status or 90 min) {int(day_met.sum())}")
    print(f"   extra trials: median {ex.median():.1f}, m.a.d. {stats.median_abs_deviation(ex.dropna()):.1f}, "
          f"negative {int((ex < 0).sum())}; without negatives median {ex[ex >= 0].median():.1f}, "
          f"m.a.d. {stats.median_abs_deviation(ex[ex >= 0].dropna()):.1f}")
    print(f"   criterion of the trigger: {s.disengaged[s.disengaged.isin([1, 2, 3])].value_counts().sort_index().to_dict()}")
print(f"\nruntime so far {(time.time() - T0) / 60:.1f} min")

**Result: not reproduced.** The project mice have 6,539 sessions, against 10,903. 5,698 sessions
met an end criterion and 841 ended earlier (13%), where the paper has 5,587 and 5,316 (49%). The
median number of extra trials is 16.0 (m.a.d. 14.0), against 19 +- 1.9 and 16 +- 1.6. Only the
count of sessions meeting a criterion is within tolerance, and the shares differ.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| cohort-counts | not reproduced |
| fig2g-days-to-proficiency | partially reproduced |
| fig2g-days-differ-across-labs | not reproduced |
| fig2s1-trials-to-proficiency | partially reproduced |
| fig2h-proportion-learned-40-days | blocked: methods underspecified |
| fig2bc-threshold-bias-iqr | partially reproduced |
| fig2-abs-bias-threshold-during-training | reproduced |
| fig2s2-variability-decreases | partially reproduced |
| early-sessions-predict-slow-learners | blocked: methods underspecified |
| fig3c-easy-errors-basic | reproduced |
| fig3de-threshold-bias-basic | reproduced |
| fig3cde-no-lab-difference-medians | blocked: methods underspecified |
| fig3cde-no-lab-difference-variances | reproduced |
| fig3s2a-trial-duration | not reproduced |
| fig3s2b-trials-per-session | partially reproduced |
| fig3f-classifier-chance-basic | blocked: other (too few mice per lab) |
| fig3f-positive-control | blocked: other (too few mice per lab) |
| fig3s3-other-classifiers | blocked: other (too few mice per lab) |
| lab-link | reproduced |
| fig3s1a-choice-consistency-basic | reproduced |
| fig3s1bc-choice-consistency-full | not reproduced |
| fig3s4-looser-criterion | not reproduced |
| fig3s5-first-full-sessions | not reproduced |
| fig3s5a-prior-used-from-first-sessions | reproduced |
| full-task-training-time | partially reproduced |
| fig4cd-biasshift-largest-at-zero | reproduced |
| fig4de-biasshift-no-lab-difference | reproduced |
| fig4f-threshold-unchanged-by-block | reproduced |
| fig4gh-lapses-change-with-block | partially reproduced |
| fig4i-bias-shifts-with-block | partially reproduced |
| fig4fi-no-lab-difference | reproduced |
| fig4j-classifier-chance-full | blocked: other (too few mice per lab) |
| fig5cf-glm-weights | partially reproduced |
| fig5dg-history-weights-higher-full | reproduced |
| fig5s2-glm-accuracy | reproduced |
| fig5s2-weights-similar-across-labs | not reproduced |
| fig5s1b-win-stay-lose-switch | reproduced |
| fig5s1c-updating-by-contrast | reproduced |
| session-end-criteria | not reproduced |

How the analysis differs from the paper's:

- The dataset holds sessions from 2019-06-11 to 2020-03-23 only; the paper's mice began training
  in January 2019. Only the 136 project mice whose first session in the dataset is a first-stage
  session are used, against 206 that started training; every count and every test has fewer mice
  than the paper's.
- The files carry no training status. Basic and full task proficiency are recomputed from the
  trials with Appendix 1, table 1d, under four readings of P1, and they agree only loosely with
  the dataset's own protocol switches (the first full-task session follows P1 by 1 to 3 days for
  29 of 64 to 40 of 57 mice).
- "50:50 sessions" are taken as basic-task sessions, as `probabilityLeft` varies from trial to
  trial in basic-task sessions.
- P1b (the first full-task session with status trained_1b) is taken as the first full-task day on
  or after P1.
- The GLM leaves out a contrast's weight for a mouse that was not shown that contrast.
- The lab classifiers were not run: they need 8 mice per lab.

What was found about the dataset:

- It holds part of the paper's data: 178 mice and 6,615 sessions from 2019-06-11 on, against 206
  mice and 10,903 sessions from January 2019.
- One recording is deposited twice, under two mice (CSHL_013 and CSHL_015, 2019-07-15).
- The training status of each session, which the paper's analyses select on, is not in the
  files.
- `probabilityLeft` is described as the block's probability of a left stimulus, but in every
  basic-task session it varies from trial to trial between 0 and 1.
- `session_start_time` is stated in UTC (+00:00) but holds local clock times.
- 257,021 trials (6%) have no stimulus onset time, and 185 sessions have none; 1,541 trials in 220
  sessions have a response before the stimulus onset (down to -5,638 s). Trial durations, and the
  1b reaction-time criterion, cannot be computed where onsets are missing.
- One session (ZM_1367, 2019-06-11) uses contrasts of 4% and 8%, which the paper does not describe.
- In 807 files the wheel position and its timestamps differ in length, and 16 files have no wheel
  series. In one session checked (ibl_witten_07), the wheel moves the opposite way to the choice on
  every trial.
- The archive's summary counts 178 subjects; one mouse (SWC_029) is not of the paper's project.